# Lab 1.1 — Multi-Agent Topology Setup (Planner–Executor & ReAct)
**Northwind Global Retail** | Microsoft Agent Framework

Build a Planner (Supervisor) that decomposes a settlement-file request and a
ReAct-style Executor that parses CSV records, wired as a state graph with
persisted state between turns.

Runs OFFLINE by default; set FOUNDRY_PROJECT_ENDPOINT + FOUNDRY_MODEL_DEPLOYMENT
in .env to run against live Azure AI Foundry models.

In [ ]:
from __future__ import annotations
import asyncio, csv, json, sys
from pathlib import Path

ROOT = next(p for p in Path(__file__).resolve().parents
            if (p / "common" / "model.py").exists())
sys.path.insert(0, str(ROOT))

from common.model import (Agent, Executor, WorkflowBuilder, WorkflowContext,
                          handler, make_chat_client, MODE)

print(f"Running in {MODE.upper()} mode")

## Step 1 — Planner Agent (Supervisor)
The Planner receives "process this settlement file" and returns a JSON plan.
In Azure mode a live LLM plans; offline, a deterministic stub emits the same
schema so downstream code is identical.

In [ ]:
client = make_chat_client()

planner = Agent(
    client=client,
    name="planner",
    instructions=(
        "You are the reconciliation supervisor for Northwind Global Retail. "
        "Given a settlement file name, respond ONLY with a JSON object with a "
        "'plan' array of steps: ingest, extract, match, post_or_escalate."
    ),
)

## Step 2 — Executor node (ReAct pattern over CSV rows)
ReAct = interleaved Reason -> Act. Here each row triggers a *reason* step
(should this row be parsed? is the promo field well-formed?) followed by an
*act* step (emit a normalized record). Note defect D5: row 9 encodes the
promo discount as an accounting-negative "(12.50)".

In [ ]:
def parse_money(raw: str) -> float:
    """Handle accounting-negative '(12.50)' -> -12.50 (planted defect D5)."""
    raw = raw.strip()
    if raw.startswith("(") and raw.endswith(")"):
        return -float(raw[1:-1])
    return float(raw)


class PlannerNode(Executor):
    """Wraps the planner agent as the graph's entry node."""
    @handler
    async def run(self, filename: str, ctx: WorkflowContext[dict]) -> None:
        resp = await planner.run(f"Plan processing for settlement file {filename}")
        plan = json.loads(resp.text)          # raises loudly if model breaks contract
        ctx.set_state("job", {"plan": plan, "file": filename})  # persisted state (key,value)
        await ctx.send_message({"file": filename, "plan": plan})


class CsvExecutorNode(Executor):
    """ReAct executor: parses raw CSV records into normalized dicts."""
    @handler
    async def run(self, task: dict, ctx: WorkflowContext[None, dict]) -> None:
        path = ROOT / "data" / "settlements" / task["file"]
        records, rejects = [], []
        with path.open() as f:
            for i, row in enumerate(csv.DictReader(f)):
                try:                                # Reason: validate
                    rec = {
                        "order_id": row["order_id"],
                        "asin": row["asin"],
                        "fba_fee": parse_money(row["fba_fee"]),
                        "promo_discount": parse_money(row["promo_discount"]),
                        "commission": parse_money(row["commission"]),
                        "net_amount": parse_money(row["net_amount"]),
                    }
                    records.append(rec)             # Act: emit
                except (ValueError, KeyError) as e:
                    rejects.append({"row": i, "error": str(e)})
        state = ctx.get_state("job") or {}
        await ctx.yield_output({"plan": state.get("plan"),
                                "parsed": len(records),
                                "rejected": rejects,
                                "records": records})

## Step 3 — Wire the state graph: Supervisor -> Executor

In [ ]:
async def main() -> dict:
    plan_node = PlannerNode(id="planner")
    exec_node = CsvExecutorNode(id="csv_executor")

    workflow = (
        WorkflowBuilder(start_executor=plan_node, name="lab1_1_topology")
        .add_edge(plan_node, exec_node)
        .build()
    )
    result = await workflow.run("settlement_2026_08_batch1.csv")
    out = result.get_outputs()[0]
    print(f"Plan steps: {[s['task'] for s in out['plan']['plan']]}")
    print(f"Parsed {out['parsed']} rows; rejected: {out['rejected']}")
    print("Sample record:", out["records"][0])
    assert out["parsed"] == 26, f"expected 26 rows, got {out['parsed']}"
    assert out["rejected"] == [], "no rows should reject once parse_money handles ()"
    assert any(r["promo_discount"] < 0 for r in out["records"]), "D5 negative promo parsed"
    print("LAB 1.1 PASS")
    return out

await main()